# Evaluate IN/OUT detections

### Convert detections to ``pd.DataFrame`` for easy use

In [77]:
import pandas as pd
import re
from datetime import datetime
from datetime import timedelta

In [59]:
# The input data
gt_path = '../data/annotations/hb_3.txt'

with open(gt_path, 'r') as f:
    data = f.read()

ground_truth = []

for line in data.split('\n'):
    # Skip the first line
    if line.startswith('time'):
        continue

    time_started = line.split(',')[0]
    time_end = line.split(',')[1]
    persons_recognized = line.split(',')[2]
    detection_type = line.split(',')[3]

    # Split the recognized persons
    persons_passed = persons_recognized.split(';')

    for person in persons_passed:
        if '[' in person or ']' in person:
            # Remove the brackets
            person = re.sub(r'\[|\]', '', person)
        
        time_str = datetime.strptime(time_started, "%H:%M")
        time_final = datetime.strptime(time_end, "%H:%M")
        
        # Convert to 24-hour format
        time_str = time_str.strftime("%H:%M")
        time_final = time_final.strftime("%H:%M")

        ground_truth.append({
            'time_started': time_str,
            'time_end': time_final,
            'person': person.strip(),
            'detection_type': detection_type.strip()
        })

In [60]:
# Convert predictions
pred_path = '../results/results_3.txt'

df_pred = pd.read_csv(pred_path, sep=',', header=None)

# Convert df_pred to the dictionary format
predictions = []
for index, row in df_pred.iterrows():
    # Ignore the first line
    if index == 0:
        continue
    
    time = row[0]
    person = row[1]
    detection_type = row[2]
    predictions.append({
        'time': datetime.strptime(time, "%H:%M").strftime("%H:%M"),
        'person': person.strip(),
        'detection_type': detection_type.strip()
    })

### Evalution

In [61]:
ground_truth

[{'time_started': '01:13',
  'time_end': '01:52',
  'person': 'Jumabek',
  'detection_type': 'IN'},
 {'time_started': '01:13',
  'time_end': '01:52',
  'person': 'Azizbek',
  'detection_type': 'IN'},
 {'time_started': '01:13',
  'time_end': '01:52',
  'person': 'Sarvar',
  'detection_type': 'IN'},
 {'time_started': '01:13',
  'time_end': '01:52',
  'person': 'Bahodirjon',
  'detection_type': 'IN'},
 {'time_started': '01:13',
  'time_end': '01:52',
  'person': 'Asadbek',
  'detection_type': 'IN'},
 {'time_started': '01:13',
  'time_end': '01:52',
  'person': 'Abdulloh',
  'detection_type': 'IN'},
 {'time_started': '01:13',
  'time_end': '01:52',
  'person': 'Otabek',
  'detection_type': 'IN'},
 {'time_started': '01:13',
  'time_end': '01:52',
  'person': 'Mirsaid',
  'detection_type': 'IN'},
 {'time_started': '01:13',
  'time_end': '01:52',
  'person': 'Oybek',
  'detection_type': 'IN'},
 {'time_started': '01:13',
  'time_end': '01:52',
  'person': 'Omadbek',
  'detection_type': 'IN'},


In [90]:
from datetime import datetime, timedelta

tp = 0
fn = 0
fp = 0

for pred in predictions:
    time = datetime.strptime(pred['time'], "%H:%M")  # keep as datetime object
    person = pred['person']
    detection_type = pred['detection_type']
    
    is_it_tp = False  # Initialize per prediction

    for gt in ground_truth:
        gt_start = datetime.strptime(gt['time_started'], "%H:%M")
        gt_end = datetime.strptime(gt['time_end'], "%H:%M")

        if person == gt['person']:
            buffer = timedelta(seconds=50)
            if gt_start - buffer <= time <= gt_end + buffer:
                if detection_type == gt['detection_type']:
                    tp += 1
                    is_it_tp = True
                    break  # Stop checking once matched

    if not is_it_tp:
        fp += 1

print(f"True Positives: {tp}")
print(f"False Positives: {fp}")

True Positives: 36
False Positives: 31


In [12]:
from datetime import timedelta
import ast

def time_str_to_seconds(t):
    """Convert MM:SS to total seconds."""
    minutes, seconds = map(int, t.strip().split(":"))
    return minutes * 60 + seconds

# Load annotation file
annotations_file = '/home/hbvision/mirsaid/face-recognition/data/annotations/hb_3.txt'
with open(annotations_file, 'r') as f:
    lines = [line.strip() for line in f.readlines() if line.strip() and not line.startswith("--")]
annotations = []
for line in lines[1:]:  # skip header
    time_start, time_end, persons_str, cam_type = line.split(',')
    if cam_type.strip() != 'IN':
        continue
    try:
        persons = [p.strip() for p in persons_str.strip('[]').split(';') if p.strip()]

    except:
        print(persons_str, "malformed")
        persons = [persons_str]

    annotations.append({
        'start': time_str_to_seconds(time_start),
        'end': time_str_to_seconds(time_end),
        'persons': set(persons)
    })

# Load system result file
results_file = '/home/hbvision/mirsaid/face-recognition/results/results_middle.txt'
with open(results_file, 'r') as f:
    lines = [line.strip() for line in f.readlines() if line.strip()]
results = []
for line in lines[1:]:  # skip header
    t, name, cam_type = line.split(',')
    if cam_type.strip() != 'IN':
        continue
    results.append({
        'time': time_str_to_seconds(t),
        'name': name.strip()
    })

# Compare annotations vs. results
matched = set()
unmatched = set()
total_gt = 0
for ann in annotations:
    ann_range = (ann['start'], ann['end'])
    for person in ann['persons']:
        total_gt += 1
        # Check if the person is detected as IN during this interval
        for r in results:
            if r['name'] == person:
                if ann_range[0] - 60 <= r['time'] <= ann_range[1] + 60:
                    matched.add((ann_range, person))  # avoid double counting
                    break
                else:
                    print(f"{person} not found in {r['name']} at {r['time']} (GT: {ann_range[0]} - {ann_range[1]})")
                    unmatched.add((ann_range, person))

# Compute True Positive Match Rate
true_positives = len(matched)
tp_match_rate = true_positives / total_gt if total_gt > 0 else 0

print(f"Total GT Persons (IN): {total_gt}")
print(f"True Positives: {true_positives}")
print(f"True Positive Match Rate: {tp_match_rate:.4f}")


Jumabek not found in Jumabek at 45 (GT: 151 - 181)
Jumabek not found in Jumabek at 60 (GT: 151 - 181)
Bahodirjon not found in Bahodirjon at 89 (GT: 151 - 181)
Uktam not found in Uktam at 87 (GT: 151 - 181)
Abdulloh not found in Abdulloh at 81 (GT: 151 - 181)
Abdulloh not found in Abdulloh at 88 (GT: 151 - 181)
Oybek not found in Oybek at 85 (GT: 151 - 181)
Dilshod not found in Dilshod at 83 (GT: 151 - 181)
Total GT Persons (IN): 30
True Positives: 30
True Positive Match Rate: 1.0000


### OUT

In [17]:
from datetime import timedelta
import ast

interval = 60

def time_str_to_seconds(t):
    """Convert MM:SS to total seconds."""
    minutes, seconds = map(int, t.strip().split(":"))
    return minutes * 60 + seconds

# Load annotation file
annotations_file = '/home/hbvision/mirsaid/face-recognition/data/annotations/hb_3.txt'
with open(annotations_file, 'r') as f:
    lines = [line.strip() for line in f.readlines() if line.strip() and not line.startswith("--")]
annotations = []
for line in lines[1:]:  # skip header
    time_start, time_end, persons_str, cam_type = line.split(',')
    if cam_type.strip() != 'OUT':
        continue
    try:
        persons = [p.strip() for p in persons_str.strip('[]').split(';') if p.strip()]

    except:
        print(persons_str, "malformed")
        persons = [persons_str]

    annotations.append({
        'start': time_str_to_seconds(time_start),
        'end': time_str_to_seconds(time_end),
        'persons': set(persons)
    })

# Load system result file
results_file = '/home/hbvision/mirsaid/face-recognition/results/results_middle.txt'
with open(results_file, 'r') as f:
    lines = [line.strip() for line in f.readlines() if line.strip()]
results = []
for line in lines[1:]:  # skip header
    t, name, cam_type = line.split(',')
    if cam_type.strip() != 'OUT':
        continue
    results.append({
        'time': time_str_to_seconds(t),
        'name': name.strip()
    })

# Compare annotations vs. results
matched = set()
unmatched = set()
total_gt = 0
for ann in annotations:
    ann_range = (ann['start'], ann['end'])
    for person in ann['persons']:
        total_gt += 1
        # Check if the person is detected as IN during this interval
        for r in results:
            if r['name'] == person:
                if ann_range[0] - interval <= r['time'] <= ann_range[1] + interval:
                    matched.add((ann_range, person))  # avoid double counting
                    break
                else:
                    print(f"{person} not found in {r['name']} at {r['time']} (GT: {ann_range[0]} - {ann_range[1]})")
                    unmatched.add((ann_range, person))

# Compute True Positive Match Rate
true_positives = len(matched)
tp_match_rate = true_positives / total_gt if total_gt > 0 else 0

print(f"Total GT Persons (OUT): {total_gt}")
print(f"True Positives: {true_positives}")
print(f"True Positive Match Rate: {tp_match_rate:.4f}")


Jumabek not found in Jumabek at 51 (GT: 113 - 148)
Jumabek not found in Jumabek at 10 (GT: 113 - 148)
Asadbek not found in Asadbek at 43 (GT: 113 - 148)
Sarvar not found in Sarvar at 49 (GT: 113 - 148)
BahodirN not found in BahodirN at 6 (GT: 113 - 148)
Uktam not found in Uktam at 35 (GT: 113 - 148)
Uktam not found in Uktam at 23 (GT: 113 - 148)
Abdulloh not found in Abdulloh at 11 (GT: 113 - 148)
Abdulloh not found in Abdulloh at 37 (GT: 113 - 148)
Oybek not found in Oybek at 34 (GT: 113 - 148)
Azizbek not found in Azizbek at 5 (GT: 113 - 148)
Total GT Persons (OUT): 28
True Positives: 25
True Positive Match Rate: 0.8929
